# Workshop Data Setup

This notebook downloads workshop datasets and loads them into Unity Catalog.

## Requirements
- Databricks workspace with **Unity Catalog enabled** (not Community Edition)
- Permission to create catalogs/schemas OR an existing catalog to use

## Datasets Available
| Dataset | Tables | Description |
|---------|--------|-------------|
| velocity_motors | 16 | Automotive dealership (Sales, CRM, Operations) |
| star_schema | 6 | Clean dimensional model |
| super_table | 1 | Messy data demo |

## 1. Setup and Environment Check

In [ ]:
# Setup and Environment Check
import os
import sys

IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IN_DATABRICKS:
    # Check for Unity Catalog support
    try:
        uc_enabled = spark.conf.get("spark.databricks.unityCatalog.enabled", "false")
        if uc_enabled.lower() != "true":
            print("WARNING: Unity Catalog may not be enabled.")
            print("This notebook requires Unity Catalog (not available in Community Edition).")
            print("Please use Databricks Free Trial or a workspace with Unity Catalog.")
    except Exception:
        pass

    # Path setup for imports
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

print("Environment ready!")
print(f"Running in Databricks: {IN_DATABRICKS}")

## 2. Configuration

Use the widgets above (in Databricks) or modify the defaults below.

In [ ]:
# Configuration Widgets
if IN_DATABRICKS:
    try:
        dbutils.widgets.removeAll()
    except Exception:
        pass

    dbutils.widgets.dropdown(
        "1_dataset", "velocity_motors", ["velocity_motors", "star_schema", "super_table", "all"], "1. Dataset"
    )
    dbutils.widgets.text("2_catalog", "workshop", "2. Target Catalog")

    print("Configure using the widgets above, then run the next cell.")
    print("")
    print("Widget options:")
    print("  1. Dataset: Choose which dataset to load")
    print("  2. Catalog: Name of the Unity Catalog to create/use")
else:
    print("Running locally - using default values.")
    print("Modify the next cell to change dataset or catalog.")

## 3. Core Functions

In [ ]:
# Core Functions for Data Setup
import io
import time
import zipfile

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Dataset configuration
DATASET_CONFIGS = {
    "velocity_motors": {
        "base_url": "https://compassagentemofiles.blob.core.windows.net/datasets/velocity_motors",
        "schemas": {
            "sales": [
                "territories",
                "salespersons",
                "vehicles",
                "features",
                "vehicle_features",
                "price_history",
                "orders",
                "order_items",
            ],
            "crm": ["customer_segments", "customers", "interactions", "leads"],
            "operations": ["warehouse_locations", "suppliers", "parts_inventory", "service_orders"],
        },
    },
    "star_schema": {
        "base_url": "https://compassagentemofiles.blob.core.windows.net/datasets/star_schema",
        "schemas": {
            "analytics": ["dim_date", "dim_product", "dim_customer", "dim_store", "dim_promotion", "fact_sales"],
        },
    },
    "super_table": {
        "base_url": "https://compassagentemofiles.blob.core.windows.net/datasets/super_table",
        "schemas": {
            "demo": ["super_table"],
        },
    },
}

# --- Bundled dataset support ---
# Zip files shipped with workshop materials in notebooks/data/
BUNDLED_ZIPS = {
    "velocity_motors": "velocity_motors_dataset.zip",
}


def _resolve_bundled_data_dir():
    """Resolve path to the notebooks/data/ directory containing bundled zip files."""
    if IN_DATABRICKS:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        notebook_dir = "/Workspace" + "/".join(notebook_path.split("/")[:-1])
        return f"{notebook_dir}/data"
    else:
        return os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")


BUNDLED_DATA_DIR = _resolve_bundled_data_dir()


def load_from_bundled_zip(dataset_name: str, table_name: str, volume_path: str) -> bool:
    """Try to load a parquet file from the bundled dataset zip.

    Args:
        dataset_name: e.g. "velocity_motors"
        table_name: e.g. "customers"
        volume_path: Destination path in UC Volume

    Returns:
        True if successful, False if zip not available or extraction failed
    """
    zip_filename = BUNDLED_ZIPS.get(dataset_name)
    if not zip_filename:
        return False

    zip_path = os.path.join(BUNDLED_DATA_DIR, zip_filename)
    if not os.path.exists(zip_path):
        return False

    parquet_filename = f"{table_name}.parquet"
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            if parquet_filename not in zf.namelist():
                print(f"    {parquet_filename} not found in bundled zip")
                return False

            content = zf.read(parquet_filename)
            print(f"    Loaded {len(content):,} bytes from bundled zip")

            if IN_DATABRICKS:
                from databricks.sdk import WorkspaceClient

                w = WorkspaceClient()
                w.files.upload(volume_path, io.BytesIO(content), overwrite=True)
                print(f"    Uploaded to {volume_path}")
            else:
                import tempfile

                temp_dir = tempfile.gettempdir()
                local_path = os.path.join(temp_dir, parquet_filename)
                with open(local_path, "wb") as f:
                    f.write(content)
                print(f"    Saved to {local_path}")

            return True
    except Exception as e:
        print(f"    Could not load from bundled zip: {e}")
        return False


# --- Download support (fallback) ---
# Create a session with retry logic for resilient downloads
_session = requests.Session()
_retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
_session.mount("https://", HTTPAdapter(max_retries=_retry))


def check_connectivity():
    """Pre-check that we can reach the Azure Blob Storage endpoint."""
    test_url = "https://compassagentemofiles.blob.core.windows.net/datasets/velocity_motors/territories.parquet"
    try:
        resp = _session.head(test_url, timeout=15)
        resp.raise_for_status()
        print("  Connectivity check: OK")
        return True
    except Exception as e:
        print(f"  Connectivity check: FAILED - {e}")
        print("")
        print("  Your workspace cannot reach the Azure Blob Storage endpoint.")
        print("  Common causes:")
        print("    - Workspace has a restrictive firewall or VNet configuration")
        print("    - Serverless compute may block outbound connections")
        print("    - Network Security Group (NSG) rules blocking Azure Blob Storage")
        print("")
        print("  Workarounds:")
        print("    1. Use a classic (non-serverless) compute cluster")
        print("    2. Ask your workspace admin to allow outbound HTTPS to:")
        print("       compassagentemofiles.blob.core.windows.net")
        print("    3. Download parquet files locally and upload via UI")
        return False


def download_file(url: str, volume_path: str) -> bool:
    """Download a file from URL to Unity Catalog Volume.

    Works with both serverless and classic compute by using the
    Databricks SDK Files API instead of local temp files.
    Includes retry logic for transient connection errors.

    Args:
        url: Source URL to download from
        volume_path: Destination path in UC Volume (e.g., /Volumes/catalog/schema/volume/file.parquet)

    Returns:
        True if successful, False otherwise
    """
    max_attempts = 3
    for attempt in range(1, max_attempts + 1):
        try:
            if attempt > 1:
                wait = 2 ** attempt
                print(f"    Retry {attempt}/{max_attempts} (waiting {wait}s)...")
                time.sleep(wait)
            else:
                print(f"    Downloading from {url}...")

            response = _session.get(url, timeout=120)
            response.raise_for_status()
            content = response.content
            print(f"    Downloaded {len(content):,} bytes")

            if IN_DATABRICKS:
                from databricks.sdk import WorkspaceClient

                w = WorkspaceClient()

                # Upload directly to volume using the Files API
                # The volume_path should be like /Volumes/catalog/schema/volume/file.parquet
                w.files.upload(volume_path, io.BytesIO(content), overwrite=True)
                print(f"    Uploaded to {volume_path}")
            else:
                # Local: just save to temp directory
                import tempfile

                temp_dir = tempfile.gettempdir()
                local_path = os.path.join(temp_dir, os.path.basename(volume_path))
                with open(local_path, "wb") as f:
                    f.write(content)
                print(f"    Saved to {local_path}")

            return True
        except Exception as e:
            if attempt == max_attempts:
                print(f"    ERROR downloading (after {max_attempts} attempts): {e}")
                return False
            # Continue to retry


def add_column_comments(catalog: str, schema: str, table: str, descriptions: dict[str, str]) -> None:
    """Add column comments to a table for Genie AI context.

    Args:
        catalog: Unity Catalog name
        schema: Schema name
        table: Table name
        descriptions: Dict mapping column names to descriptions
    """
    full_table = f"{catalog}.{schema}.{table}"

    for column, description in descriptions.items():
        # Escape single quotes in description
        safe_desc = description.replace("'", "''")
        try:
            # Use backticks for column names to handle reserved words
            sql = f"ALTER TABLE {full_table} ALTER COLUMN `{column}` COMMENT '{safe_desc}'"
            spark.sql(sql)
        except Exception:
            # Column might not exist - that's okay
            pass


def setup_dataset(dataset_name: str, catalog: str) -> dict[str, bool]:
    """Set up a complete dataset in Unity Catalog.

    Tries to load from bundled zip first, falls back to Azure Blob download.

    Args:
        dataset_name: Name of the dataset (velocity_motors, star_schema, super_table)
        catalog: Target Unity Catalog name

    Returns:
        Dict mapping table names to success status
    """
    config = DATASET_CONFIGS.get(dataset_name)
    if not config:
        print(f"ERROR: Unknown dataset: {dataset_name}")
        return {}

    base_url = config["base_url"]
    results = {}

    print(f"\n{'=' * 60}")
    print(f"Setting up: {dataset_name}")
    print(f"Target catalog: {catalog}")
    print(f"{'=' * 60}")

    # Check if bundled data is available for this dataset
    has_bundled = dataset_name in BUNDLED_ZIPS and os.path.exists(
        os.path.join(BUNDLED_DATA_DIR, BUNDLED_ZIPS[dataset_name])
    )
    if has_bundled:
        print(f"Bundled data found for {dataset_name} -- will use local zip first")

    # Try to load column descriptions
    try:
        from config.dataset_schemas import get_column_descriptions

        has_descriptions = True
    except ImportError:
        has_descriptions = False
        print("Note: Column descriptions not available (config.dataset_schemas not found)")

    for schema_name, tables in config["schemas"].items():
        # Create schema
        print(f"\nCreating schema: {catalog}.{schema_name}")
        try:
            spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")
            print("  Schema ready!")
        except Exception as e:
            print(f"  ERROR creating schema: {e}")
            continue

        # Load each table
        for table in tables:
            print(f"\n  Loading table: {table}")

            try:
                volume_path = f"/Volumes/{catalog}/staging/uploads/{dataset_name}_{table}.parquet"
                full_table = f"{catalog}.{schema_name}.{table}"

                # Try bundled zip first, fall back to download
                loaded = load_from_bundled_zip(dataset_name, table, volume_path)
                if not loaded:
                    file_url = f"{base_url}/{table}.parquet"
                    if not download_file(file_url, volume_path):
                        results[table] = False
                        continue

                # Create table from parquet
                spark.sql(f"DROP TABLE IF EXISTS {full_table}")
                spark.sql(f"""
                    CREATE TABLE {full_table}
                    AS SELECT * FROM read_files('{volume_path}', format => 'parquet')
                """)

                print(f"    Table created: {full_table}")

                # Add column comments if available
                if has_descriptions:
                    try:
                        # Map dataset schemas to our schema structure
                        ds_schema = schema_name
                        if dataset_name in ["star_schema", "super_table"]:
                            ds_schema = "default"

                        descriptions = get_column_descriptions(dataset_name, ds_schema, table)
                        add_column_comments(catalog, schema_name, table, descriptions)
                        print(f"    Added {len(descriptions)} column comments")
                    except Exception as e:
                        print(f"    Note: Could not add column comments: {e}")

                results[table] = True

            except Exception as e:
                print(f"    ERROR: {e}")
                results[table] = False

    return results


print("Core functions loaded!")

## 4. Execute Data Setup

This cell will:
1. Create the catalog and staging volume
2. Download datasets from Azure Blob Storage
3. Load them into Unity Catalog tables
4. Add column comments for Genie AI

In [ ]:
# Execute Data Setup

# Get configuration from widgets or defaults
if IN_DATABRICKS:
    dataset = dbutils.widgets.get("1_dataset")
    catalog = dbutils.widgets.get("2_catalog")
else:
    # Default values for local testing
    dataset = "velocity_motors"
    catalog = "workshop"

print("Configuration:")
print(f"  Dataset: {dataset}")
print(f"  Catalog: {catalog}")
print()

if not IN_DATABRICKS:
    print("NOTICE: Not running in Databricks. Skipping actual data load.")
    print("This notebook is designed to run in Databricks with Unity Catalog.")
else:
    # Pre-check: can we reach Azure Blob Storage? (used as fallback if bundled data unavailable)
    can_download = check_connectivity()
    if not can_download:
        print("\n  Bundled data will be used where available. Azure download unavailable as fallback.")

    # Bootstrap: Create catalog, staging schema, and volume
    print("Creating catalog and staging volume...")
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
        print(f"  Catalog '{catalog}' ready")
    except Exception as e:
        print(f"  Could not create catalog (may need permissions or already exists): {e}")
        print("  Tip: Try using an existing catalog you have access to.")

    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.staging")
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.staging.uploads")
        print(f"  Staging volume '{catalog}.staging.uploads' ready")
    except Exception as e:
        print(f"  Could not create staging volume: {e}")

    # Determine which datasets to load
    if dataset == "all":
        datasets_to_load = ["velocity_motors", "star_schema", "super_table"]
    else:
        datasets_to_load = [dataset]

    # Load each dataset
    all_results = {}
    for ds in datasets_to_load:
        results = setup_dataset(ds, catalog)
        all_results[ds] = results

    # Summary
    print("\n" + "=" * 60)
    print("SETUP SUMMARY")
    print("=" * 60)

    total_success = 0
    total_failed = 0

    for ds, results in all_results.items():
        success = sum(1 for v in results.values() if v)
        failed = sum(1 for v in results.values() if not v)
        total_success += success
        total_failed += failed

        print(f"\n{ds}: {success} succeeded, {failed} failed")
        for table, ok in results.items():
            status = "OK" if ok else "FAILED"
            print(f"  [{status}] {table}")

    print(f"\nTotal: {total_success} tables loaded, {total_failed} failed")

## 5. Verify Data

Check that tables were loaded correctly with row counts.

In [ ]:
# Verify Data - Show row counts for loaded tables

if IN_DATABRICKS:
    print("\nData Verification - Row Counts")
    print("=" * 60)

    for ds_name, ds_config in DATASET_CONFIGS.items():
        if dataset != "all" and ds_name != dataset:
            continue

        print(f"\n{ds_name.upper()}:")

        for schema_name, tables in ds_config["schemas"].items():
            for table in tables:
                full_table = f"{catalog}.{schema_name}.{table}"
                try:
                    count = spark.sql(f"SELECT COUNT(*) FROM {full_table}").collect()[0][0]
                    print(f"  {schema_name}.{table}: {count:,} rows")
                except Exception as e:
                    print(f"  {schema_name}.{table}: ERROR - {e}")
else:
    print("Verification skipped - not running in Databricks.")

## 6. Next Steps: Create Your Genie Space

Now that your data is loaded, set up a Genie Space for AI-powered analytics!

In [ ]:
# Next Steps

if IN_DATABRICKS:
    catalog_name = dbutils.widgets.get("2_catalog")
else:
    catalog_name = "workshop"

print(f"""
=== Next Steps: Create Your Genie Space ===

1. Click 'Genie' in the left sidebar
2. Click 'New Genie Space'
3. Add tables from: {catalog_name}.*
4. Configure instructions (see docs/GENIE_BEST_PRACTICES.md)

Recommended tables for Velocity Motors demo:
  - {catalog_name}.sales.orders
  - {catalog_name}.sales.order_items
  - {catalog_name}.sales.vehicles
  - {catalog_name}.sales.salespersons
  - {catalog_name}.crm.customers
  - {catalog_name}.crm.customer_segments

For the "Dirty vs Clean" demo:
  - Create one space with: {catalog_name}.demo.super_table (no instructions)
  - Create another with: {catalog_name}.analytics.* (with instructions)

Data setup complete.
""")

## 7. Cleanup (Optional)

Uncomment and run the cell below to remove all workshop data.

In [ ]:
# CLEANUP - Uncomment to remove workshop data
# WARNING: This will delete all data loaded by this notebook!

# if IN_DATABRICKS:
#     catalog_name = dbutils.widgets.get("2_catalog")
#
#     print(f"WARNING: About to drop catalog '{catalog_name}' and all its contents!")
#     print("Uncomment the next line to proceed:")
#
#     # spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE")
#     # print(f"Catalog '{catalog_name}' dropped.")
# else:
#     print("Cleanup skipped - not running in Databricks.")